# Notebook 1: Physics-Based GPU Fleet Telemetry Simulator

**GPU Fleet Autopilot — Research & Simulation Suite**

This notebook implements the physics-grounded telemetry simulator for large-scale GPU clusters (100 to 10,000 GPUs). It models:
- **Dynamic Power Draw**: Idle vs dynamic training workloads ($P_{\text{idle}} = 150\text{W}$, $P_{\text{max}} = 700\text{W}$ for H100)
- **Newton's Cooling Law ODE Integration**: $\tau \frac{dT_{\text{phys}}}{dt} = -(T_{\text{phys}} - T_{\text{amb}}) + R_{\text{thermal}} P(t)$
- **Thermal Throttling Hysteresis**: Clock throttling engages at $92^\circ\text{C}$ down to $500\text{MHz}$, recovering at $83^\circ\text{C}$
- **Canonical Telemetry Schema v1.0**: Emits standard NVIDIA DCGM field conventions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

# Styling
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)
np.random.seed(42)
print("Environment initialized successfully.")

## 1. Physics Engine Implementation

We model each GPU die with thermal inertia $\tau = 25.0\text{s}$ and nominal thermal resistance $R_{\text{thermal}} = 0.075\,^\circ\text{C}/\text{W}$.

In [ ]:
class GpuSimulator:
    def __init__(self, gpu_id, node_id, rack_id, model="H100"):
        self.gpu_id = gpu_id
        self.node_id = node_id
        self.rack_id = rack_id
        self.model = model
        
        # Thermal state
        self.ambient_temp = 24.0 + np.random.uniform(-1.0, 1.0)
        self.temp_phys = 65.0 + np.random.uniform(-2.0, 2.0)
        self.tau = 25.0
        self.r_thermal = 0.075
        self.is_throttled = False
        
        # Metrics
        self.utilization = 10.0
        self.power = 250.0
        self.sm_clock = 1980
        self.ecc_sbe = 0
        self.ecc_dbe = 0
        self.xid = 0
        self.nvlink_errors = 0
        self.net_errors = 0
        self.perf_ratio = 1.0

    def step(self, dt=2.0, is_busy=True):
        # Workload phases: Compute burst (95%), Sync (45%), Checkpoint (15%)
        if is_busy:
            u = np.clip(np.random.normal(92.0, 4.0), 0.0, 100.0)
        else:
            u = np.clip(np.random.normal(5.0, 2.0), 0.0, 100.0)
        self.utilization = u
        
        # Dynamic power draw
        p_idle = 150.0
        p_dyn = 550.0
        self.power = np.clip(p_idle + p_dyn * (u / 100.0) + np.random.normal(0, 5.0), 100.0, 750.0)
        
        # Euler ODE step for Newton's cooling
        d_temp = (dt / self.tau) * (-(self.temp_phys - self.ambient_temp) + self.r_thermal * self.power)
        self.temp_phys = np.clip(self.temp_phys + d_temp, 15.0, 115.0)
        
        # Thermal throttling hysteresis
        if self.temp_phys >= 92.0:
            self.is_throttled = True
        elif self.temp_phys <= 83.0:
            self.is_throttled = False
            
        if self.is_throttled:
            self.sm_clock = int(max(500, 1980 - 15 * (self.temp_phys - 83.0)))
            self.perf_ratio = self.sm_clock / 1980.0
        else:
            self.sm_clock = 1980
            self.perf_ratio = 1.0
            
        # Observed sensor reading with noise
        observed_temp = np.clip(self.temp_phys + np.random.normal(0, 1.0), 15.0, 115.0)
        return {
            "gpu_id": self.gpu_id,
            "node_id": self.node_id,
            "dcgm_gpu_temp": observed_temp,
            "dcgm_power_usage": self.power,
            "dcgm_gpu_utilization": self.utilization,
            "dcgm_sm_clock": self.sm_clock,
            "performance_ratio": self.perf_ratio
        }

## 2. Simulate Multi-Hour Workload Trace

In [ ]:
sim = GpuSimulator("gpu-00042", "node-006", "rack-01")
records = []

# Simulate 600 ticks (20 minutes of 2s ticks)
for t in range(600):
    rec = sim.step(dt=2.0, is_busy=(t > 60))
    rec["time_s"] = t * 2.0
    records.append(rec)

df = pd.DataFrame(records)
print(f"Generated {len(df)} telemetry samples.")
df.head()

## 3. Visualize Power & Thermal ODE Dynamics

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.set_xlabel("Time (seconds)")
ax1.set_ylabel("Temperature (°C)", color="tab:red")
ax1.plot(df["time_s"], df["dcgm_gpu_temp"], color="tab:red", label="GPU Temperature")
ax1.axhline(92, color="darkred", linestyle="--", label="Throttle Limit (92°C)")
ax1.tick_params(axis="y", labelcolor="tab:red")

ax2 = ax1.twinx()
ax2.set_ylabel("Power (Watts)", color="tab:blue")
ax2.plot(df["time_s"], df["dcgm_power_usage"], color="tab:blue", alpha=0.5, label="Power Usage")
ax2.tick_params(axis="y", labelcolor="tab:blue")

plt.title("Physics Simulator: Newton's Cooling ODE with Dynamic Workload Response")
fig.tight_layout()
plt.show()